# Shooting Method for the Blasius Equation
**AE801 Scientific Computing – Lab 1**

Blasius equation: $f''' + \tfrac12 f f'' = 0$, with $f(0)=0,\ f'(0)=0,\ f'(\infty)=1$.

**Approach**
1. Write as a first-order system $Y=(f,f',f'')^T$, $Y'=G(Y)=(f',\,f'',\,-\tfrac12 f f'')^T$.
2. The missing initial condition is $s=f''(0)$.
3. Integrate to $\eta_\infty$ with fixed-step RK3 / RK4 and define the residual $F(s)=f'(\eta_\infty;s)-1$.
4. Solve $F(s)=0$ with the secant method.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ETA_INF = 10.0
TOL = 1e-12
S_REF = 0.33205733621519   # literature value, used only to measure error

## Step (a): first-order system and RK stepper
Both RK methods are written through their Butcher tableaus ($A$, $b$), so one stepper serves both.

In [ ]:
def rhs(Y):
    # Y = [f, f', f'']
    return np.array([Y[1], Y[2], -0.5 * Y[0] * Y[2]])

TABLEAU = {
    "RK3": (np.array([[0, 0, 0],
                      [0.5, 0, 0],
                      [-1.0, 2.0, 0]]),
            np.array([1/6, 4/6, 1/6])),
    "RK4": (np.array([[0, 0, 0, 0],
                      [0.5, 0, 0, 0],
                      [0, 0.5, 0, 0],
                      [0, 0, 1.0, 0]]),
            np.array([1/6, 2/6, 2/6, 1/6])),
}

def rk_step(Y, h, A, b):
    k = []
    for i in range(len(b)):
        k.append(rhs(Y + h * sum(A[i, j] * k[j] for j in range(i))))
    return Y + h * sum(b[i] * k[i] for i in range(len(b)))

def integrate(s, h, method, store=False):
    A, b = TABLEAU[method]
    n = int(round(ETA_INF / h))
    Y = np.array([0.0, 0.0, s])
    if store:
        out = np.empty((n + 1, 3)); out[0] = Y
    for i in range(n):
        Y = rk_step(Y, h, A, b)
        if store:
            out[i + 1] = Y
    return out if store else Y

## Steps (b), (c): shooting parameter, residual and secant method
$s=f''(0)$ is the unknown; residual $F(s)=f'(\eta_\infty;s)-1$.

In [ ]:
def residual(s, h, method):
    return integrate(s, h, method)[1] - 1.0

def secant(h, method, s0=0.2, s1=0.5, tol=TOL, itmax=60):
    F0, F1 = residual(s0, h, method), residual(s1, h, method)
    for it in range(itmax):
        s2 = s1 - F1 * (s1 - s0) / (F1 - F0)
        s0, F0 = s1, F1
        s1, F1 = s2, residual(s2, h, method)
        if abs(s1 - s0) < tol:
            return s1, it + 1
    raise RuntimeError("secant did not converge")

## Steps (d), (f): grid refinement for RK3 and RK4
The observed order is computed from successive solutions, $p\approx\log_2\frac{|s_{h}-s_{h/2}|}{|s_{h/2}-s_{h/4}|}$, so it does not depend on the reference value.

In [ ]:
steps = [0.4, 0.2, 0.1, 0.05, 0.025]
results = {}
for m in ("RK3", "RK4"):
    print(f"\n=== {m} (eta_inf = {ETA_INF}) ===")
    print(f"{'dEta':>8} {'f2(0)':>16} {'|s - s_ref|':>14} {'iters':>6} {'obs.order':>10}")
    S, E = [], []
    for h in steps:
        s, its = secant(h, m)
        S.append(s); E.append(abs(s - S_REF))
        p = ""
        if len(S) >= 3:
            p = f"{np.log2(abs(S[-2]-S[-3]) / abs(S[-1]-S[-2])):.2f}"
        print(f"{h:8.3f} {s:16.10f} {E[-1]:14.3e} {its:6d} {p:>10}")
    results[m] = (np.array(S), np.array(E))
    print(f"Check (g): f''(0) = {S[-1]:.3f}  (expected ~0.332)")

### Log–log convergence plot

In [ ]:
hs = np.array(steps)
plt.figure(figsize=(6.5, 4.8))
for m, mk in (("RK3", "o-"), ("RK4", "s-")):
    plt.loglog(hs, results[m][1], mk, label=m)
    slope = np.polyfit(np.log(hs[:4]), np.log(results[m][1][:4]), 1)[0]
    print(f"{m}: fitted slope over coarse 4 steps = {slope:.2f}")
plt.loglog(hs, 0.05 * hs**3, "k--", lw=0.8, label=r"$O(\Delta\eta^3)$")
plt.loglog(hs, 0.01 * hs**4, "k:", lw=0.8, label=r"$O(\Delta\eta^4)$")
plt.xlabel(r"$\Delta\eta$"); plt.ylabel(r"$|f''(0)-f''_{ref}|$")
plt.grid(True, which="both", alpha=0.3); plt.legend()
plt.title("Convergence of the shooting solution")
plt.tight_layout(); plt.show()

**Note:** the RK4 error levels off near $10^{-9}$. This floor comes from truncating the domain at $\eta_\infty=10$ (not from RK4), so the plot flattens there. A larger $\eta_\infty$ lowers the floor.

## Step (e): profiles of $f$, $f'$, $f''$

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
for a, m in zip(ax, ("RK3", "RK4")):
    h = 0.025
    s, _ = secant(h, m)
    sol = integrate(s, h, m, store=True)
    eta = np.linspace(0, ETA_INF, len(sol))
    a.plot(eta, sol[:, 0], label="f")
    a.plot(eta, sol[:, 1], label="f'")
    a.plot(eta, sol[:, 2], label="f''")
    a.set_xlim(0, 8); a.set_xlabel(r"$\eta$")
    a.set_title(f"{m}, $\\Delta\\eta$ = {h}")
    a.grid(alpha=0.3); a.legend()
plt.tight_layout(); plt.show()

## Conclusions
- Converged $f''(0)\approx 0.33206$, matching the expected $0.332$ to three digits.
- RK3 shows an observed order close to 3, and RK4 approaches 4 until the $\eta_\infty$ truncation error dominates.
- The secant method converged in about 6 iterations for every step size.